# 04 - Track B: Two-Step Revenue Optimization

This notebook implements a two-step approach: demand prediction + revenue optimization.

## Configuration


In [1]:
# Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import joblib

# Import our utilities
import sys
sys.path.append('.')
from utils_splits import create_time_split, validate_no_leakage

# Set random seed
np.random.seed(42)

# Project paths
PROJECT_ROOT = Path.cwd().parent
FEATURES_DIR = PROJECT_ROOT / "features"
RESULTS_DIR = PROJECT_ROOT / "results"
CODE_DIR = PROJECT_ROOT / "code"

TEST_START_DATE = "2024-07-01"
print(f"Two-step revenue optimization for Sicily Airbnb")


Two-step revenue optimization for Sicily Airbnb


## Load Data and Prepare Features


In [ ]:
# Load training table
print("Loading training data...")
training_table = pd.read_parquet(FEATURES_DIR / 'training_table.parquet')

print(f"✅ Loaded training data:")
print(f"  Shape: {training_table.shape}")
print(f"  Date range: {training_table['date'].min()} to {training_table['date'].max()}")
print(f"  Booking rate: {training_table['booked'].mean():.2%}")

# Sample data for faster training (reduce to 100k rows for classification)
if len(training_table) > 100000:
    print("Sampling data for faster training...")
    training_table = training_table.sample(n=100000, random_state=42)
    print(f"  Sampled shape: {training_table.shape}")

# Check for missing values
missing_counts = training_table.isnull().sum()
if missing_counts.sum() > 0:
    print(f"\n⚠️  Missing values found:")
    print(missing_counts[missing_counts > 0])
else:
    print(f"\n✅ No missing values in training data")


In [ ]:
def prepare_features(df, target_col='booked'):
    """
    Prepare features for machine learning models (same as notebook 03).
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with all features
    target_col : str
        Target column name ('booked' for classification, 'price' for regression)
        
    Returns:
    --------
    X : pd.DataFrame
        Feature matrix
    y : pd.Series
        Target variable
    feature_names : list
        List of feature names
    """
    # Select features for modeling (same as price regression)
    feature_columns = [
        # Seasonality features
        'day_of_week', 'is_weekend', 'month', 'week_of_year', 'day_of_year',
        'is_holiday', 'is_holiday_window', 'is_peak_season', 'is_ferragosto',
        'fourier_sin_1', 'fourier_cos_1', 'fourier_sin_2', 'fourier_cos_2',
        'is_spring', 'is_summer', 'is_autumn', 'is_winter',
        
        # Listing features
        'accommodates', 'bedrooms', 'bathrooms', 'amenities_count',
        'number_of_reviews', 'review_scores_rating', 'minimum_nights',
        'maximum_nights', 'availability_365',
        
        # Tourism intensity (if available)
        'migration_ratio',
        
        # Weather features (stubbed)
        'temp_avg', 'precip_mm', 'humidity', 'wind_speed'
    ]
    
    # Filter to available columns
    available_features = [col for col in feature_columns if col in df.columns]
    print(f"Using {len(available_features)} features for modeling")
    
    # Create feature matrix
    X = df[available_features].copy()
    y = df[target_col].copy()
    
    # Handle categorical features
    categorical_features = ['room_type', 'neighbourhood', 'neighbourhood_cleansed']
    categorical_features = [col for col in categorical_features if col in df.columns]
    
    if categorical_features:
        print(f"Categorical features: {categorical_features}")
        for col in categorical_features:
            if df[col].nunique() > 50:  # High cardinality
                print(f"  Dropping high-cardinality feature: {col} ({df[col].nunique()} unique values)")
                if col in X.columns:
                    X = X.drop(col, axis=1)
            else:
                # One-hot encode low-cardinality categoricals
                dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
                X = pd.concat([X, dummies], axis=1)
                print(f"  One-hot encoded: {col} -> {dummies.shape[1]} features")
    
    # Fill any remaining missing values
    X = X.fillna(0)
    
    return X, y, X.columns.tolist()


In [ ]:
# Create time-based train/test split
print("Creating time-based split...")
train_df, test_df = create_time_split(training_table, TEST_START_DATE)
validate_no_leakage(train_df, test_df)

print(f"\n📊 Split summary:")
print(f"  Train booking rate: {train_df['booked'].mean():.2%}")
print(f"  Test booking rate: {test_df['booked'].mean():.2%}")


## Step 1: Demand Classification Models


In [ ]:
# Prepare features for demand classification
print("Preparing features for demand classification...")
X_train, y_train_booked, feature_names = prepare_features(train_df, target_col='booked')
X_test, y_test_booked, _ = prepare_features(test_df, target_col='booked')

print(f"\n✅ Feature preparation completed:")
print(f"  Train features: {X_train.shape}")
print(f"  Test features: {X_test.shape}")
print(f"  Feature names: {len(feature_names)}")

# Check class balance
print(f"\n📊 Class distribution:")
print(f"  Train - Booked: {y_train_booked.sum():,} ({y_train_booked.mean():.2%})")
print(f"  Train - Available: {(~y_train_booked).sum():,} ({(~y_train_booked).mean():.2%})")
print(f"  Test - Booked: {y_test_booked.sum():,} ({y_test_booked.mean():.2%})")
print(f"  Test - Available: {(~y_test_booked).sum():,} ({(~y_test_booked).mean():.2%})")

assert X_train.isnull().sum().sum() == 0, "Train features contain missing values"
assert X_test.isnull().sum().sum() == 0, "Test features contain missing values"


In [ ]:
# Define classification models
from sklearn.model_selection import RandomizedSearchCV

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
}

# Hyperparameter search spaces
param_grids = {
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5, 10],
        'class_weight': ['balanced', None]
    },
    'XGBoost': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 10],
        'learning_rate': [0.01, 0.1, 0.2],
        'scale_pos_weight': [1, (1 - y_train_booked.mean()) / y_train_booked.mean()]
    }
}

# Train and evaluate models
demand_results = {}
best_demand_model = None
best_auc = 0

print("Training demand classification models...")
for name, model in models.items():
    print(f"\n🔧 Training {name}...")
    
    # Hyperparameter tuning for tree-based models
    if name in param_grids:
        print(f"  Hyperparameter tuning...")
        search = RandomizedSearchCV(
            model, param_grids[name], 
            n_iter=10, cv=3, random_state=42, n_jobs=-1, 
            scoring='roc_auc'
        )
        search.fit(X_train, y_train_booked)
        model = search.best_estimator_
        print(f"  Best params: {search.best_params_}")
    
    # Train model
    model.fit(X_train, y_train_booked)
    
    # Make predictions
    y_pred_proba_train = model.predict_proba(X_train)[:, 1]
    y_pred_proba_test = model.predict_proba(X_test)[:, 1]
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calculate metrics
    train_auc = roc_auc_score(y_train_booked, y_pred_proba_train)
    test_auc = roc_auc_score(y_test_booked, y_pred_proba_test)
    train_f1 = f1_score(y_train_booked, y_pred_train)
    test_f1 = f1_score(y_test_booked, y_pred_test)
    
    # Store results
    demand_results[name] = {
        'model': model,
        'train_auc': train_auc,
        'test_auc': test_auc,
        'train_f1': train_f1,
        'test_f1': test_f1,
        'y_pred_proba_test': y_pred_proba_test
    }
    
    print(f"  Train AUC: {train_auc:.4f}, F1: {train_f1:.4f}")
    print(f"  Test AUC: {test_auc:.4f}, F1: {test_f1:.4f}")
    
    # Track best model
    if test_auc > best_auc:
        best_auc = test_auc
        best_demand_model = model
        best_demand_name = name

print(f"\n🏆 Best demand model: {best_demand_name} (Test AUC: {best_auc:.4f})")


In [ ]:
# Save best demand model
demand_model_path = PROJECT_ROOT / 'best_demand_model.pkl'
joblib.dump(best_demand_model, demand_model_path)
print(f"✅ Best demand model saved to: {demand_model_path}")

# Save metrics
demand_metrics = {
    name: {
        'train_auc': float(res['train_auc']),
        'test_auc': float(res['test_auc']),
        'train_f1': float(res['train_f1']),
        'test_f1': float(res['test_f1'])
    }
    for name, res in demand_results.items()
}

import json
metrics_path = RESULTS_DIR / 'metrics_demand_classification.json'
with open(metrics_path, 'w') as f:
    json.dump(demand_metrics, f, indent=2)
print(f"✅ Demand metrics saved to: {metrics_path}")

# Print classification report for best model
print(f"\n📊 Classification Report - {best_demand_name}:")
print(classification_report(y_test_booked, demand_results[best_demand_name]['model'].predict(X_test)))


## Step 2: Revenue Optimization


In [ ]:
# Load the best price prediction model
price_model_path = PROJECT_ROOT / 'best_price_model_updated.pkl'
if not price_model_path.exists():
    price_model_path = PROJECT_ROOT / 'best_price_model.pkl'

print(f"Loading price prediction model from: {price_model_path}")
price_model = joblib.load(price_model_path)
print(f"✅ Price model loaded: {type(price_model).__name__}")

# Get booking probabilities for test set
booking_proba_test = demand_results[best_demand_name]['y_pred_proba_test']
print(f"\n📊 Booking probability statistics:")
print(f"  Mean: {booking_proba_test.mean():.4f}")
print(f"  Min: {booking_proba_test.min():.4f}")
print(f"  Max: {booking_proba_test.max():.4f}")
print(f"  Std: {booking_proba_test.std():.4f}")


In [ ]:
def optimize_revenue(features_row, price_model, demand_model, price_range=(10, 500), n_prices=50):
    """
    Find the price that maximizes expected revenue.
    
    Expected Revenue = Price × P(booking | price, features)
    
    Parameters:
    -----------
    features_row : pd.Series or dict
        Single row of features (without price)
    price_model : sklearn model
        Trained price prediction model
    demand_model : sklearn model
        Trained demand classification model
    price_range : tuple
        (min_price, max_price) to search
    n_prices : int
        Number of prices to evaluate
        
    Returns:
    --------
    optimal_price : float
        Price that maximizes expected revenue
    max_revenue : float
        Maximum expected revenue
    revenue_curve : dict
        Price-revenue pairs
    """
    # Create array of prices to test
    prices = np.linspace(price_range[0], price_range[1], n_prices)
    
    # Prepare base features (without price)
    base_features = pd.DataFrame([features_row] * n_prices)
    
    # For each price, predict booking probability
    # Note: In reality, booking probability might depend on price
    # For now, we use the predicted probability based on other features
    # In a full implementation, you'd need price as a feature in the demand model
    
    # Get base booking probability (without price effect)
    booking_proba_base = demand_model.predict_proba(base_features)[:, 1]
    
    # Simple price elasticity: higher price -> lower booking probability
    # This is a simplified model - in practice you'd train demand model with price
    base_price = price_model.predict(base_features)[0] if hasattr(price_model, 'predict') else features_row.get('price', 50)
    price_elasticity = -0.3  # For every 10% price increase, ~3% decrease in booking prob
    
    revenues = []
    for price in prices:
        # Adjust booking probability based on price relative to predicted price
        price_ratio = price / max(base_price, 1)
        adjusted_proba = booking_proba_base[0] * (1 + price_elasticity * np.log(price_ratio))
        adjusted_proba = np.clip(adjusted_proba, 0, 1)  # Ensure valid probability
        
        expected_revenue = price * adjusted_proba
        revenues.append(expected_revenue)
    
    revenues = np.array(revenues)
    optimal_idx = np.argmax(revenues)
    optimal_price = prices[optimal_idx]
    max_revenue = revenues[optimal_idx]
    
    revenue_curve = dict(zip(prices, revenues))
    
    return optimal_price, max_revenue, revenue_curve

print("✅ Revenue optimization function defined")


In [ ]:
# Apply revenue optimization to a sample of test data
print("Applying revenue optimization to test sample...")
sample_size = min(1000, len(test_df))
test_sample = test_df.sample(n=sample_size, random_state=42).copy()
test_sample_features, _, _ = prepare_features(test_sample, target_col='booked')

print(f"Optimizing revenue for {sample_size} listings...")

revenue_recommendations = []
for idx, (row_idx, features_row) in enumerate(test_sample_features.iterrows()):
    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx + 1}/{sample_size}...")
    
    try:
        optimal_price, max_revenue, revenue_curve = optimize_revenue(
            features_row, price_model, best_demand_model
        )
        
        # Get predicted base price
        predicted_price = price_model.predict(pd.DataFrame([features_row]))[0]
        
        # Get booking probability
        booking_proba = best_demand_model.predict_proba(pd.DataFrame([features_row]))[0, 1]
        
        # Actual values from test set
        actual_price = test_sample.loc[row_idx, 'price']
        actual_booked = test_sample.loc[row_idx, 'booked']
        
        revenue_recommendations.append({
            'listing_id': test_sample.loc[row_idx, 'listing_id'],
            'date': test_sample.loc[row_idx, 'date'],
            'actual_price': actual_price,
            'predicted_price': predicted_price,
            'optimal_price': optimal_price,
            'price_difference': optimal_price - actual_price,
            'price_change_pct': (optimal_price - actual_price) / actual_price * 100,
            'booking_probability': booking_proba,
            'expected_revenue': max_revenue,
            'actual_booked': actual_booked,
            'actual_revenue': actual_price if actual_booked else 0
        })
    except Exception as e:
        print(f"  Error for row {row_idx}: {e}")
        continue

revenue_df = pd.DataFrame(revenue_recommendations)
print(f"\n✅ Revenue optimization completed for {len(revenue_df)} listings")


In [ ]:
# Analyze revenue optimization results
print("\n📊 Revenue Optimization Results:")
print(f"  Sample size: {len(revenue_df):,}")
print(f"\n💶 Price Statistics:")
print(f"  Actual price - Mean: €{revenue_df['actual_price'].mean():.2f}, Median: €{revenue_df['actual_price'].median():.2f}")
print(f"  Predicted price - Mean: €{revenue_df['predicted_price'].mean():.2f}, Median: €{revenue_df['predicted_price'].median():.2f}")
print(f"  Optimal price - Mean: €{revenue_df['optimal_price'].mean():.2f}, Median: €{revenue_df['optimal_price'].median():.2f}")
print(f"  Average price change: €{revenue_df['price_difference'].mean():.2f} ({revenue_df['price_change_pct'].mean():.1f}%)")

print(f"\n📈 Revenue Impact:")
actual_total_revenue = revenue_df['actual_revenue'].sum()
expected_optimal_revenue = revenue_df['expected_revenue'].sum()
print(f"  Actual revenue (observed bookings): €{actual_total_revenue:,.2f}")
print(f"  Expected revenue (with optimal prices): €{expected_optimal_revenue:,.2f}")
print(f"  Potential improvement: €{expected_optimal_revenue - actual_total_revenue:,.2f} ({((expected_optimal_revenue - actual_total_revenue) / max(actual_total_revenue, 1) * 100):.1f}%)")

print(f"\n🎯 Booking Probability:")
print(f"  Mean: {revenue_df['booking_probability'].mean():.4f}")
print(f"  Range: [{revenue_df['booking_probability'].min():.4f}, {revenue_df['booking_probability'].max():.4f}]")

# Save recommendations
revenue_output_path = RESULTS_DIR / 'revenue_recommendations_sample.csv'
revenue_df.to_csv(revenue_output_path, index=False)
print(f"\n✅ Revenue recommendations saved to: {revenue_output_path}")


In [ ]:
# Visualize revenue optimization results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price comparison
axes[0, 0].scatter(revenue_df['actual_price'], revenue_df['optimal_price'], alpha=0.5)
axes[0, 0].plot([revenue_df['actual_price'].min(), revenue_df['actual_price'].max()], 
                [revenue_df['actual_price'].min(), revenue_df['actual_price'].max()], 
                'r--', label='No change')
axes[0, 0].set_xlabel('Actual Price (€)')
axes[0, 0].set_ylabel('Optimal Price (€)')
axes[0, 0].set_title('Actual vs Optimal Price')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Price change distribution
axes[0, 1].hist(revenue_df['price_change_pct'], bins=50, edgecolor='black')
axes[0, 1].axvline(0, color='r', linestyle='--', label='No change')
axes[0, 1].set_xlabel('Price Change (%)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Recommended Price Changes')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Booking probability vs optimal price
axes[1, 0].scatter(revenue_df['optimal_price'], revenue_df['booking_probability'], alpha=0.5)
axes[1, 0].set_xlabel('Optimal Price (€)')
axes[1, 0].set_ylabel('Booking Probability')
axes[1, 0].set_title('Optimal Price vs Booking Probability')
axes[1, 0].grid(True, alpha=0.3)

# Expected revenue distribution
axes[1, 1].hist(revenue_df['expected_revenue'], bins=50, edgecolor='black')
axes[1, 1].set_xlabel('Expected Revenue (€)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Expected Revenue')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = RESULTS_DIR / 'revenue_optimization_analysis.png'
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✅ Revenue analysis plot saved to: {plot_path}")
plt.show()


## Summary

This notebook implements a two-step revenue optimization approach:

1. **Step 1 - Demand Prediction**: Train classification models to predict booking probability
   - Models tested: Logistic Regression, Random Forest, XGBoost
   - Best model selected based on Test AUC score

2. **Step 2 - Revenue Optimization**: Find optimal prices that maximize expected revenue
   - Expected Revenue = Price × P(booking | features, price)
   - Applied to test sample of listings
   - Uses price elasticity model to adjust booking probability based on price

**Outputs:**
- `best_demand_model.pkl`: Trained demand classification model
- `metrics_demand_classification.json`: Classification performance metrics
- `revenue_recommendations_sample.csv`: Price optimization recommendations
- `revenue_optimization_analysis.png`: Visualization of results
